In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from dataclasses import dataclass
from typing import Optional, Dict, Any

from sklearn.metrics import mean_absolute_error

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)

In [2]:
MODEL_NAME = "FacebookAI/roberta-large"
TRAIN_CSV = "/content/ielts_train_df.csv"
VAL_CSV   = "/content/ielts_val_df.csv"
TEST_CSV  = "/content/ielts_test_df.csv"

MAX_LENGTH = 384
SEED = 42
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 1e-6
EPOCHS = 10
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./deberta_ielts_4criteria"

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

set_seed(SEED)

In [3]:
# --- CẬP NHẬT CELL 3: LÀM SẠCH DỮ LIỆU TRIỆT ĐỂ ---

# 1. Load dữ liệu gốc
train_df = pd.read_csv(TRAIN_CSV, engine="python", on_bad_lines="skip")
val_df = pd.read_csv(VAL_CSV, engine="python", on_bad_lines="skip") if os.path.exists(VAL_CSV) else None
test_df = pd.read_csv(TEST_CSV, engine="python", on_bad_lines="skip") if os.path.exists(TEST_CSV) else None

# 2. Tách tập validation nếu chưa có
if val_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

needed_cols = ["prompt", "essay"] + TARGET_COLS

def robust_clean_df(df):
    if df is None: return None
    # Chỉ lấy các cột cần thiết
    df = df[needed_cols].copy()

    # Ép kiểu nhãn sang numeric và loại bỏ NaN ở nhãn
    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

    # QUAN TRỌNG: Ép kiểu essay sang string và loại bỏ khoảng trắng thừa
    df['essay'] = df['essay'].astype(str).str.strip()

    # Loại bỏ các bài viết quá ngắn (dưới 20 ký tự thường là dữ liệu lỗi)
    df = df[df['essay'].str.len() > 20].reset_index(drop=True)

    # Giới hạn điểm số trong dải hợp lệ [0.0, 9.0] để tránh outlier làm nổ gradient
    for col in TARGET_COLS:
        df[col] = df[col].clip(0.0, 9.0)

    return df

# Áp dụng hàm làm sạch cho các tập dữ liệu
train_df = robust_clean_df(train_df)
val_df = robust_clean_df(val_df)
test_df = robust_clean_df(test_df)

print(f"Train shape: {train_df.shape if train_df is not None else 0}")
print(f"Val shape: {val_df.shape if val_df is not None else 0}")
print(train_df[TARGET_COLS].head())

Train shape: (6618, 6)
Val shape: (827, 6)
    TR   CC   LR  GRA
0  6.0  6.0  6.0  6.0
1  6.5  6.5  6.5  6.5
2  7.0  7.0  7.0  7.0
3  5.0  5.0  5.0  5.0
4  4.5  4.5  4.5  4.5


In [4]:
def build_input_text(row):
    prompt = str(row["prompt"]).strip()
    essay = str(row["essay"]).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

train_df["text"] = train_df.apply(build_input_text, axis=1)
val_df["text"] = val_df.apply(build_input_text, axis=1)

if test_df is not None:
    test_df["text"] = test_df.apply(build_input_text, axis=1)

def make_labels(df):
    df = df.copy()
    df["labels"] = (df[TARGET_COLS] / 9.0).values.tolist()
    return df

train_df = make_labels(train_df)
val_df = make_labels(val_df)
if test_df is not None:
    test_df = make_labels(test_df)

In [5]:
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

if test_df is not None:
    test_ds = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
    dataset_dict["test"] = test_ds

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)
# QUAN TRỌNG: Chỉ giữ lại các cột số (input_ids, attention_mask, labels)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format(type="torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/6618 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

In [7]:
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        return sum_embeddings / sum_mask

class DebertaForMultiRegression(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 4, dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.pooler = MeanPooling() # Dùng lớp pooling chuẩn

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, num_labels)

        # Khởi tạo head với bias=6.0 để giữ ổn định lúc đầu
        nn.init.normal_(self.head.weight, std=0.01)
        nn.init.constant_(self.head.bias, 0.65)

        self.loss_fn = nn.SmoothL1Loss() # Source chuyên nghiệp thường dùng cái này thay vì Huber

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.pooler(outputs.last_hidden_state, attention_mask)
        logits = self.head(self.dropout(pooled_output))
        if torch.isnan(outputs.last_hidden_state).any() or torch.isinf(outputs.last_hidden_state).any():
            raise ValueError("NaN/Inf detected in backbone output")
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            print("Logits min/max:", logits.min().item(), logits.max().item())
            raise ValueError("NaN/Inf detected in logits")

        loss = None
        if labels is not None:
            #print("Labels min/max:", labels.min().item(), labels.max().item())
            loss = self.loss_fn(logits, labels.float())

            if torch.isnan(loss).any() or torch.isinf(loss).any():
                #print("Logits min/max:", logits.min().item(), logits.max().item())
                #print("Labels min/max:", labels.min().item(), labels.max().item())
                raise ValueError("NaN/Inf detected in loss")

        return {"loss": loss, "logits": logits}

model = DebertaForMultiRegression(MODEL_NAME)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class RegressionCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # nếu labels đã là tensor thì stack
        if isinstance(features[0]["labels"], torch.Tensor):
            labels = torch.stack([f["labels"] for f in features]).float()
        else:
            labels = torch.tensor([f["labels"] for f in features], dtype=torch.float)

        batch = self.tokenizer.pad(
            [{k: v for k, v in f.items() if k != "labels"} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        return batch

data_collator = RegressionCollator(tokenizer)

In [9]:
from sklearn.metrics import cohen_kappa_score

def round_to_half(x):
    return np.round(x * 2) / 2


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    if np.isnan(preds).any():
        print("WARNING: preds contains NaN")

    preds = preds * 9.0
    labels = labels * 9.0

    preds = np.clip(preds, 0.0, 9.0)
    labels = np.clip(labels, 0.0, 9.0)

    preds_half = round_to_half(preds)

    maes = [mean_absolute_error(labels[:, i], preds_half[:, i]) for i in range(4)]

    qwks = []
    for i in range(4):
        y_true = (labels[:, i] * 2).astype(int)
        y_pred = (preds_half[:, i] * 2).astype(int)
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        qwks.append(score)

    return {
        "mean_mae": np.mean(maes),
        "mean_qwk": np.mean(qwks),
        "tr_qwk": qwks[0],
        "cc_qwk": qwks[1],
        "lr_qwk": qwks[2],
        "gra_qwk": qwks[3],
        "within_0.5_acc": np.mean(np.abs(preds_half - labels) <= 0.5),
    }

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0,
    load_best_model_at_end=True,
    metric_for_best_model="mean_qwk",
    greater_is_better=True,
    fp16=False,
    bf16=True,
    report_to="none",
    save_total_limit=2,
    max_grad_norm=0.5,           # CHẶN GRADIENT TẠI ĐÂY
    gradient_checkpointing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
# --- SỬA CELL 10 ---
from transformers import get_cosine_schedule_with_warmup

def get_optimizer_params(model, encoder_lr, decoder_lr, weight_decay=0.0):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_parameters = [
        {'params': [p for n, p in model.backbone.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.backbone.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters() if "head" in n and not any(nd in n for nd in no_decay)],
         'lr': decoder_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if "head" in n and any(nd in n for nd in no_decay)],
         'lr': decoder_lr, 'weight_decay': 0.0},
    ]
    return optimizer_parameters

# Tính toán số bước (Step)
num_training_steps = (len(tokenized_ds["train"]) // BATCH_SIZE) * EPOCHS

# Cấu hình LR tối ưu: 2e-5 cho backbone và 5e-5 cho head
optimizer_parameters = get_optimizer_params(
    model,
    encoder_lr=5e-6,
    decoder_lr=1e-5,
    weight_decay=0.01
)
optimizer = torch.optim.AdamW(optimizer_parameters)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps
)

In [12]:
# --- SỬA CELL 11 ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    # Ép kiểu tuple rõ ràng để tránh lỗi "cannot unpack"
    optimizers=(optimizer, scheduler),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

In [13]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(debug_loader))
print(batch.keys())
print(batch["labels"])
print(batch["labels"].shape)
print(batch["labels"].dtype)

KeysView({'input_ids': tensor([[    0, 10975,  4454,  3765, 10311,   742, 50118, 10787,   749,    32,
          1408,    10,  1307,  1280,     9,   418,    15,  3117,    49,  6117,
             7,   185,   233,    11,   103,  3612,  1612,  9150,     4,  5763,
          5848,    14,    24,    74,    28,   357,   114,   209,   749,    64,
          1930,     5,   418,    15,   408,     7,   185,   233,    11,  1612,
             4,   598,    99,  5239,   109,    47,  2854,    50, 11967,   116,
         50118, 50118, 10975,  1723, 46937,   742, 50118,   970,    32,   319,
             9,   749,    11,     5,   232,    54,    32,   634,   372,  1280,
             9,   418,    15, 20170,     9,    49,   893,     4,    38,  2854,
            14,    51,   197,   386,  1408,   418,    15,   408,  2414,  1713,
            11,    14,   169,    51,    40,   386,     5, 18197,     9,   419,
          5612,    11,    49,   301,  1567,   426,     4,     7,   386,    19,
           920,  1535,  1612,

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mean Mae,Mean Qwk,Tr Qwk,Cc Qwk,Lr Qwk,Gra Qwk,Within 0.5 Acc
1,0.043722,0.009265,0.985943,0.257030,0.262210,0.266187,0.245209,0.254514,0.439843
2,0.033310,0.008018,0.908253,0.412008,0.435863,0.424889,0.398789,0.388490,0.475816
3,0.032060,0.007247,0.858374,0.458937,0.469386,0.466807,0.465770,0.433786,0.501511
4,0.026154,0.007112,0.837062,0.503627,0.509610,0.507587,0.522979,0.474330,0.527207
5,0.022857,0.006896,0.838573,0.507825,0.517063,0.513004,0.522052,0.479181,0.507860
6,0.017773,0.007097,0.848247,0.494427,0.503993,0.497340,0.510003,0.466373,0.508767
7,0.013839,0.007359,0.861548,0.497150,0.512036,0.498562,0.499392,0.478611,0.502721
8,0.011268,0.007233,0.844015,0.537757,0.558858,0.543860,0.555425,0.492887,0.522370
9,0.009426,0.007330,0.853839,0.504911,0.517053,0.516660,0.514147,0.471785,0.522068
10,0.007141,0.007249,0.852781,0.503942,0.512426,0.506277,0.520532,0.476531,0.522068


TrainOutput(global_step=4140, training_loss=0.023821949058972693, metrics={'train_runtime': 3858.1554, 'train_samples_per_second': 17.153, 'train_steps_per_second': 1.073, 'total_flos': 0.0, 'train_loss': 0.023821949058972693, 'epoch': 10.0})

In [15]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:", val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
    print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.007232637144625187, 'eval_mean_mae': 0.844014510512352, 'eval_mean_qwk': 0.5377573832624439, 'eval_tr_qwk': 0.5588576764921185, 'eval_cc_qwk': 0.5438599635213814, 'eval_lr_qwk': 0.5554251356811213, 'eval_gra_qwk': 0.49288675735515464, 'eval_within_0.5_acc': 0.5223700120918985, 'eval_runtime': 20.5932, 'eval_samples_per_second': 40.159, 'eval_steps_per_second': 40.159, 'epoch': 10.0}


early stopping required metric_for_best_model, but did not find eval_mean_qwk so early stopping is disabled


Test metrics: {'test_loss': 0.007263470441102982, 'test_mean_mae': 0.8396739214658737, 'test_mean_qwk': 0.5429731268390046, 'test_tr_qwk': 0.5606839011960052, 'test_cc_qwk': 0.5507984692142667, 'test_lr_qwk': 0.5610495231350047, 'test_gra_qwk': 0.4993606138107416, 'test_within_0.5_acc': 0.523852657004831, 'test_runtime': 21.3436, 'test_samples_per_second': 38.794, 'test_steps_per_second': 38.794, 'epoch': 10.0}


In [16]:
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")

('./deberta_ielts_4criteria/best_model/tokenizer_config.json',
 './deberta_ielts_4criteria/best_model/tokenizer.json')

In [17]:
from google.colab import runtime
runtime.unassign()